Explore the data

In [1]:
import pandas as pd
import numpy as np

#Load CSV file
df = pd.read_csv("../data/Orders_with_issues.csv")

#Check sample data 
df.head((20))

#Check data types
df.info()

#Check nulls 
df.isna().sum()

#Get unique null companies 

df['ShippingCompany'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          239 non-null    float64
 1   CustomerID       240 non-null    object 
 2   OrderDate        237 non-null    object 
 3   ShippedDate      219 non-null    object 
 4   ShippingCost     231 non-null    object 
 5   ShipCountry      245 non-null    object 
 6   ShipCity         241 non-null    object 
 7   ShippingCompany  245 non-null    object 
dtypes: float64(1), object(7)
memory usage: 15.4+ KB


FedEx Logistics                   54
DHL Express                       53
Aramex International              51
UPS Worldwide                     46
Kiwilytics Goods Shipping LLC.    41
Name: ShippingCompany, dtype: int64

Cleaning the data 


In [2]:
#Convert OrderDate and ShipDate to datetime
df["OrderDate"] = pd.to_datetime(df["OrderDate"],errors="coerce")
df["ShippedDate"] = pd.to_datetime(df["ShippedDate"],errors="coerce")


#Clean Shipping cost 
df["ShippingCost"] = pd.to_numeric(df["ShippingCost"],errors="coerce")
df.loc[df["ShippingCost"] <0 ,"ShippingCost"] = np.nan
df["ShippingCost"] = df["ShippingCost"].fillna(df["ShippingCost"].median())

#Drop the rows with missing order date and ship date
df = df[~(df['OrderDate'].isna() | df['ShippedDate'].isna() )]

#Handling nulls 
df["OrderID"] = df["OrderID"].ffill()
df["CustomerID"] = df["CustomerID"].fillna("Unknown")
df["ShipCity"] = df["ShipCity"].fillna("Unspecified")

#Standardize names

df["ShipCountry"] = df["ShipCountry"].str.strip().str.title()
df["ShipCity"] = df["ShipCity"].str.strip().str.title()
df["ShippingCompany"] = df["ShippingCompany"].str.strip()

#Fix specific case

df.loc[df["ShippingCompany"].str.contains("Kiwilytics",na=False),"ShippingCompany"] = "Kiwilytics Goods Shipping LLC."

df

,OrderID,CustomerID,OrderDate,ShippedDate,ShippingCost,ShipCountry,ShipCity,ShippingCompany
0,1000.0,C001,2025-05-17,2025-07-30,234.09,Germany,Hamburg,Kiwilytics Goods Shipping LLC.
1,1001.0,C002,2025-01-26,2025-07-30,320.61,Canada,Montreal,UPS Worldwide
2,1002.0,C003,2025-03-08,2025-07-30,165.17,Canada,Vancouver,FedEx Logistics
3,1003.0,C004,2025-03-24,2025-07-30,12.55,Germany,Munich,Aramex International
4,1004.0,C005,2025-04-15,2025-07-30,186.36,Canada,Vancouver,FedEx Logistics
...,...,...,...,...,...,...,...,...
239,1239.0,C030,2025-07-22,2025-07-30,340.32,Uk,Liverpool,Kiwilytics Goods Shipping LLC.
240,1240.0,C001,2024-11-19,2025-07-30,372.98,Uk,Manchester,Aramex International
242,1242.0,C003,2025-01-16,2025-07-30,241.16,Usa,New York,Kiwilytics Goods Shipping LLC.
243,1243.0,C004,2024-09-16,2025-07-30,83.54,Egypt,Cairo,DHL Express


Add Feature Engineering


In [3]:
#Days between order and shipping 

df["DeliveryDays"] = (df["ShippedDate"] - df["OrderDate"]).dt.days

def getStatus(row):
    if pd.isna(row):
        return "Unknown"
    elif row > 15:
        return "Late"
    else:
        return "OnTime"
df["DeliveryStatus"] = df["DeliveryDays"].apply(getStatus)

#Domestic vs International
domestic_countries = ["Germany"]
def check_domestic(country):
    if country in domestic_countries:
        return "Yes"
    else:
        return "No"
df["Domestic"] = df["ShipCountry"].apply(check_domestic)
df.head(20)

,OrderID,CustomerID,OrderDate,ShippedDate,ShippingCost,ShipCountry,ShipCity,ShippingCompany,DeliveryDays,DeliveryStatus,Domestic
0,1000.0,C001,2025-05-17,2025-07-30,234.09,Germany,Hamburg,Kiwilytics Goods Shipping LLC.,74,Late,Yes
1,1001.0,C002,2025-01-26,2025-07-30,320.61,Canada,Montreal,UPS Worldwide,185,Late,No
2,1002.0,C003,2025-03-08,2025-07-30,165.17,Canada,Vancouver,FedEx Logistics,144,Late,No
3,1003.0,C004,2025-03-24,2025-07-30,12.55,Germany,Munich,Aramex International,128,Late,Yes
4,1004.0,C005,2025-04-15,2025-07-30,186.36,Canada,Vancouver,FedEx Logistics,106,Late,No
5,1005.0,C006,2024-12-24,2025-07-30,74.93,Usa,New York,UPS Worldwide,218,Late,No
7,1007.0,C008,2024-08-10,2025-07-30,209.94,Germany,Berlin,Aramex International,354,Late,Yes
8,1008.0,C009,2025-02-06,2025-07-30,234.09,Usa,Houston,DHL Express,174,Late,No
10,1010.0,C011,2024-12-03,2025-07-30,16.05,Germany,Unspecified,Aramex International,239,Late,Yes
11,1011.0,C012,2024-10-08,2025-07-30,391.79,Canada,Vancouver,DHL Express,295,Late,No


Get shipping company average cost   

In [4]:
avg_shipping_byCompany = df.groupby("ShippingCompany")["ShippingCost"].mean()
print(avg_shipping_byCompany) 

ShippingCompany
Aramex International              265.410682
DHL Express                       241.496429
FedEx Logistics                   214.283913
Kiwilytics Goods Shipping LLC.    244.623939
UPS Worldwide                     245.797368
Name: ShippingCost, dtype: float64


Export the data and show breif of the data 

In [12]:
#Export the data to csv file 
df.to_csv("../output/Cleaned_orders.csv",index=False,mode="w")

#Get final dataset snapshot
print(df.head())

#Get delivery status breakdown
print(df["DeliveryStatus"].value_counts())

#Get Orders by Country 
print(df["ShipCountry"].value_counts())
#Get average days to deliver to countries 
print(df.groupby("ShipCountry")["DeliveryDays"].mean())

#Get Average cost by country city

print(df.groupby(["ShipCountry","ShipCity"])["ShippingCost"].mean())

#Get Order by City 
print(avg_shipping_byCompany) 
print(df.value_counts(["ShipCountry","ShipCity"]))

#Get Top 3 shipping companies
print(df["ShippingCompany"].value_counts().head(3))



   OrderID CustomerID  OrderDate ShippedDate  ShippingCost ShipCountry  \
0   1000.0       C001 2025-05-17  2025-07-30        234.09     Germany   
1   1001.0       C002 2025-01-26  2025-07-30        320.61      Canada   
2   1002.0       C003 2025-03-08  2025-07-30        165.17      Canada   
3   1003.0       C004 2025-03-24  2025-07-30         12.55     Germany   
4   1004.0       C005 2025-04-15  2025-07-30        186.36      Canada   

    ShipCity                 ShippingCompany  DeliveryDays DeliveryStatus  \
0    Hamburg  Kiwilytics Goods Shipping LLC.            74           Late   
1   Montreal                   UPS Worldwide           185           Late   
2  Vancouver                 FedEx Logistics           144           Late   
3     Munich            Aramex International           128           Late   
4  Vancouver                 FedEx Logistics           106           Late   

  Domestic  
0      Yes  
1       No  
2       No  
3      Yes  
4       No  
Late      198
